# FeatherLens Stage 1B-1: official pretrained GPU preflight
Run the cells in order on an actual Colab or Kaggle GPU. This notebook calls the
shared Python implementation; it performs no optimizer updates or model selection.
Local CPU tests and configured weight IDs are not official-weight/GPU evidence.
See `docs/gpu_preflight.md` in the published checkout for the full setup and return procedure.
The first returned T4 report verified all official weights and FP32 checks; ConvNeXt BF16 failed.
Complete acceptance and the corrected automatic AMP selection still require successful GPU evidence.

## 1. Environment and runtime detection
Colab: **Runtime → Change runtime type → Hardware accelerator → GPU → Save**.
Kaggle: **Settings → Accelerator → GPU**, and enable **Internet** for the initial online run.
The available device and quota vary; do not assume a T4. Inspect the installed stack
first. Do not install the repository's CPU requirements over a working CUDA stack.

In [ ]:
import importlib.metadata
import importlib.util
import json
import os
from pathlib import Path
import sys

print({"python": sys.version.split()[0], "platform": sys.platform})
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is missing: select a GPU runtime and install a compatible CUDA Torch/Torchvision pair deliberately.")
import torch
try:
    import torchvision
except Exception as error:
    raise RuntimeError("Installed Torch/Torchvision are incompatible; repair the accelerator stack deliberately, restart, then rerun.") from error
print({"torch": torch.__version__, "torchvision": torchvision.__version__,
       "cuda_build": torch.version.cuda, "cuda_available": torch.cuda.is_available(),
       "device_count": torch.cuda.device_count()})
if torch.cuda.is_available():
    print({"device_name": torch.cuda.get_device_name(0),
           "total_memory_bytes": torch.cuda.get_device_properties(0).total_memory})
else:
    print("GPU execution is PENDING. The runner will export pending_gpu without downloading weights.")

## 2. Repository setup at a published commit
Copy the full 40-character SHA of the published Stage 1B-1 commit from the PR.
Do not use a branch name or the older Stage 1A SHA. The notebook cannot know the
SHA of the future commit that contains it. The report records the actual checked-out HEAD.
For a private repository, set `PRIVATE_REPOSITORY=True`: an optional fine-grained
read-only GitHub token is entered with `getpass`, passed through a temporary
`GIT_ASKPASS` helper, and never added to URLs, commands, remotes, or output.
Alternatively upload/attach a clean checkout **including `.git`** at the published SHA
and set `REPO_DIR` to it. A ZIP without Git provenance is rejected.

In [ ]:
def checkout_repository(root, url, commit, private=False):
    import getpass
    import os
    from pathlib import Path
    import re
    import subprocess
    import sys
    import tempfile
    from urllib.parse import urlsplit

    if not isinstance(commit, str) or re.fullmatch(r"[0-9a-fA-F]{40}", commit) is None:
        raise ValueError("Set the full 40-character published commit SHA before repository setup.")
    parsed = urlsplit(url)
    if parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError("Repository URL must be credential-free HTTPS without query or fragment.")
    root = Path(root).resolve()
    if root.exists():
        if not (root / ".git").exists():
            raise ValueError("Existing source must include .git provenance; use a fresh path or a clean pinned checkout.")
    else:
        env = dict(os.environ)
        env["GIT_TERMINAL_PROMPT"] = "0"
        token = None
        try:
            with tempfile.TemporaryDirectory(prefix="featherlens-askpass-") as temporary:
                if private:
                    if os.name != "posix":
                        raise RuntimeError("Private clone helper targets Colab/Kaggle Linux; attach a clean pinned Git checkout on other platforms.")
                    token = getpass.getpass("Read-only GitHub token (hidden): ")
                    if not token:
                        raise ValueError("A nonempty read-only token is required for this private clone.")
                    helper = Path(temporary) / "askpass.py"
                    helper.write_text("#!" + sys.executable + "\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['FEATHERLENS_GIT_TOKEN'])\n", encoding="utf-8")
                    helper.chmod(0o700)
                    env["GIT_ASKPASS"] = str(helper)
                    env["FEATHERLENS_GIT_TOKEN"] = token
                # No credentials in command arguments; suppress Git's remote error text.
                subprocess.run(["git", "-c", "credential.helper=", "clone", "--no-checkout", url, str(root)],
                               env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
                subprocess.run(["git", "-C", str(root), "checkout", "--detach", commit],
                               env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        except subprocess.CalledProcessError:
            raise RuntimeError("Repository setup failed. Check access and the published SHA; remove only the incomplete clone before retrying.") from None
        finally:
            env.pop("FEATHERLENS_GIT_TOKEN", None)
            env.pop("GIT_ASKPASS", None)
            token = None
    actual = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
    if actual.lower() != commit.lower():
        raise ValueError("Existing checkout differs from the published commit; attach/check out the intended commit explicitly.")
    dirty = subprocess.check_output(["git", "-C", str(root), "status", "--porcelain", "--untracked-files=no"], text=True)
    if dirty.strip():
        raise ValueError("Use a clean published checkout; tracked files have local modifications.")
    return actual

CLOUD = "colab"  # Choose "colab" or "kaggle" explicitly.
if CLOUD not in {"colab", "kaggle"}:
    raise ValueError("CLOUD must be colab or kaggle")
WORK_DIR = Path("/content" if CLOUD == "colab" else "/kaggle/working")
REPO_DIR = WORK_DIR / "featherlens"
REPO_URL = "https://github.com/AnonyBOSS/featherlens-pixels-to-language.git"
PUBLISHED_COMMIT = ""  # REQUIRED: paste the published Stage 1B-1 commit SHA.
PRIVATE_REPOSITORY = True  # False for public access or an already attached checkout.
OFFLINE = False  # Online Colab first; set before section 3 imports Hugging Face.
HF_CACHE_DIR = None  # Optional staged dataset-only cache root, never a personal cache with credentials.
if HF_CACHE_DIR is not None:
    os.environ["HF_HOME"] = str(HF_CACHE_DIR)
if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
SOURCE_COMMIT = checkout_repository(REPO_DIR, REPO_URL, PUBLISHED_COMMIT, private=PRIVATE_REPOSITORY)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print({"source_commit": SOURCE_COMMIT})

## 3. Dependency verification
Preserve installed CUDA Torch/Torchvision. Missing non-Torch data dependencies are
listed below; installation requires changing the explicit consent flag.
Existing versions are retained and the final versions are recorded in the report.
If an installed stack fails an import or `pip check`, repair deliberately and restart.

In [ ]:
import subprocess

INSTALL_MISSING_DATA_DEPS = False  # Deliberate opt-in, only if packages below are missing.
data_dependencies = {"datasets": ("datasets", "5.1.0"), "numpy": ("numpy", "2.5.4"),
                     "PIL": ("Pillow", "13.0.0"), "fsspec": ("fsspec", "2026.7.0")}
missing = [f"{package}=={version}" for module, (package, version) in data_dependencies.items()
           if importlib.util.find_spec(module) is None]
print({"missing_non_torch_dependencies": missing})
if missing and not INSTALL_MISSING_DATA_DEPS:
    raise RuntimeError("Review missing packages, opt in with INSTALL_MISSING_DATA_DEPS=True, then rerun this cell.")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *missing], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print({package: importlib.metadata.version(package) for package, _ in data_dependencies.values()})
from src.training.runtime import PreflightConfig, collect_runtime, run_gpu_preflight, validate_preflight_preparation
from src.training.weights import resolve_weight_info
from src.models.pretrained import ModelConfig, get_model_spec

print(json.dumps(collect_runtime(REPO_DIR), indent=2))

## 4. Dataset preparation and source accessibility
Supply the original `preparation.json` and `classification_manifest.jsonl` in
`PREPARED_DIR`. The small handoff ZIP has **metadata only, no images**. Optionally
set `PREPARED_ZIP` to extract exactly one copy of each known file; all other entries
are ignored. Files are never overwritten. Do not run the audit or prepare another split.
The runner accesses only pinned Hugging Face **official train** through the existing
module, after canonical metadata validation. Train/validation membership is reused;
official test pixels are never requested. The first online run needs network and disk.
For offline staging, see the guide; it has not been verified in a fresh Kaggle session.
Mount Drive here if desired, before configuring persistence and running section 8.

In [ ]:
def extract_prepared_metadata(archive, destination):
    from pathlib import Path, PurePosixPath
    import zipfile

    destination = Path(destination)
    names = ("preparation.json", "classification_manifest.jsonl")
    with zipfile.ZipFile(archive) as handle:
        selected = {}
        for name in names:
            matches = [entry for entry in handle.infolist()
                       if not entry.is_dir() and PurePosixPath(entry.filename).name == name]
            if len(matches) != 1:
                raise ValueError("Handoff ZIP must contain exactly one copy of each known preparation file.")
            selected[name] = matches[0]
        if any((destination / name).exists() for name in names):
            raise FileExistsError("Prepared metadata already exists; use it directly or choose an empty directory.")
        # Read only known entries, never extract archive paths or arbitrary files.
        payloads = {name: handle.read(entry) for name, entry in selected.items()}
    destination.mkdir(parents=True, exist_ok=True)
    for name, payload in payloads.items():
        with (destination / name).open("xb") as output:
            output.write(payload)

MOUNT_GOOGLE_DRIVE = False
if CLOUD == "colab" and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

PREPARED_DIR = WORK_DIR / "classification-v1"  # Or /content/drive/MyDrive/... or /kaggle/input/<dataset>/...
PREPARED_ZIP = None  # Example: Path("/content/handoff.zip"); no image data is inside.
if PREPARED_ZIP is not None:
    extract_prepared_metadata(PREPARED_ZIP, PREPARED_DIR)

# OFFLINE and HF_CACHE_DIR were configured in section 2, before Hugging Face imports.
# Offline mode needs the exact pinned train snapshot and official weights staged separately.
OUTPUT_DIR = REPO_DIR / "outputs" / "gpu-preflight" / __import__("uuid").uuid4().hex
PERSISTENT_DIR = None  # Optional new run directory under mounted Drive, e.g. /content/drive/MyDrive/featherlens/<unique-run>
# Kaggle: /kaggle/working survives only when outputs are saved in a notebook version.
BATCH_SIZE = 2
NUM_WORKERS = 0

## 5. Manifest and class-label validation
Validate the existing preparation before any source fetch. Required canonical
manifest SHA: `a8bd05e048febf32df6d84a6e1bc16eddf6416be723e12dfc89621044962736b`.
Expected seed 42, 4,794 train / 1,200 validation / 5,794 test assignments and 200
ordered classes. Reading test **metadata** here does not load test images.

In [ ]:
prepared_identity = validate_preflight_preparation(PREPARED_DIR)
print(json.dumps({key: value for key, value in prepared_identity.items() if key != "class_names"}, indent=2))
print({"ordered_class_count": len(prepared_identity["class_names"])})

## 6. Official pretrained-weight loading
These IDs and official URLs are configuration metadata, not proof of loaded weights.
Section 8 calls the verified builder once per architecture, reuses Torch Hub's
checkpoint cache, checks the file hash, and compares every retained tensor and buffer.
Absent/corrupt offline weights fail clearly. No random-weight fallback is allowed.
To stage weights, use only the three files resolved below in
`Path(torch.hub.get_dir()) / "checkpoints"`; set `torch.hub.set_dir(...)` first if needed.

In [ ]:
TORCH_HUB_DIR = None  # Optional staged hub root containing checkpoints/<official filenames>.
if TORCH_HUB_DIR is not None:
    torch.hub.set_dir(str(TORCH_HUB_DIR))
architectures = ("convnext_tiny", "efficientnet_b3", "regnet_y_3_2gf")
weight_identities = [resolve_weight_info(ModelConfig(architecture=name)) for name in architectures]
print(json.dumps(weight_identities, indent=2))
print({"torch_hub_checkpoints": str(Path(torch.hub.get_dir()) / "checkpoints")})

## 7. Model construction and feature extraction
The shared runner constructs the existing adapters with genuine verified weights.
At the unchanged full-image fit/white 224×224 baseline, expect logits `[B,200]`,
spatial `[B,D,7,7]`, pooled `[B,D]`. Native weight crop recipes remain metadata;
ImageNet normalization runs exactly once and flips apply only to training.
The following is an expected shape contract; executed shapes appear in the report.

In [ ]:
print({name: {"expected_logits": [BATCH_SIZE, 200],
              "expected_spatial": [BATCH_SIZE, get_model_spec(name).encoder_channels, 7, 7],
              "expected_pooled": [BATCH_SIZE, get_model_spec(name).encoder_channels]}
       for name in architectures})

## 8. Actual GPU inference and backward-pass smoke tests
This is the single execution call. It validates source mapping, fetches representative
train/validation batches, then checks each model sequentially and releases GPU state.
Frozen, partial and full policies run FP32 and supported AMP checks with finite loss.
Training batches verify gradients and frozen masks; validation uses eval/inference_mode
without backward. There is no `optimizer.step` or training run.
Partial failures are retained in the exported report. Without CUDA, status is
`pending_gpu` and all official-weight/GPU checks remain unexecuted.

In [ ]:
config = PreflightConfig(prepared_dir=PREPARED_DIR, output_dir=OUTPUT_DIR,
                         persistent_dir=PERSISTENT_DIR,
                         architectures=("convnext_tiny", "efficientnet_b3", "regnet_y_3_2gf"),
                         batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, seed=42,
                         offline=OFFLINE, warmup_batches=2, measured_batches=5, amp="auto")
report = run_gpu_preflight(config)
print({"status": report["status"], "output_dir": str(OUTPUT_DIR)})

## 9. CUDA and AMP results
Inspect actual per-policy precision outcomes. Unsupported precision, overflow,
failure and not-run statuses must remain visible. Gradients are checked after AMP
unscaling; AMP backward is a smoke test, not a completed training run.
Automatic AMP uses native BF16 capability (excluding emulation), otherwise FP16.
The first T4 run exposed emulation-only BF16 selection; inspect the new native-capability flag.

In [ ]:
for model_result in report.get("models", []):
    print(json.dumps({"architecture": model_result["architecture"],
                      "status": model_result["status"],
                      "checks": model_result.get("checks", []),
                      "failures": model_result.get("failures", [])}, indent=2))

## 10. Memory and throughput measurements
The runner reports peak allocated/reserved GPU bytes per executed check. Throughput
is warmed, synchronized **inference on a resident representative batch**, separate
from backward memory and end-to-end data loading. Read the measured batch size and
device from the report; do not compare models or invent absent measurements.

In [ ]:
for model_result in report.get("models", []):
    print(json.dumps({"architecture": model_result["architecture"],
                      "throughput": model_result.get("throughput"),
                      "check_memory": [{"policy": check.get("policy"), "precision": check.get("precision"),
                                        "memory": check.get("memory")}
                                       for check in model_result.get("checks", [])]}, indent=2))

## 11. Persistent-storage verification
If `PERSISTENT_DIR` was set before section 8, inspect the runner's write/readback
evidence. A mounted Drive write/read does **not** prove durability in a fresh session.
Kaggle `/kaggle/working` is session storage until **Save Version** saves its outputs;
download the evidence now and verify it again after reopening the saved version.
Do not copy source images, model weights or credential caches into the evidence bundle.

In [ ]:
print(json.dumps(report.get("persistence", {}), indent=2))
print(json.dumps(report.get("artifacts", {}), indent=2))

## 12. Results summary and evidence export
Return **`preflight-report.json` and `preflight-evidence.zip`** from the printed run
directory (the ZIP includes its compact hash inventory). Check the JSON for all three
architectures, all policies, genuine checkpoint hashes, actual CUDA device and failures.
In Colab use the Files pane to download, or deliberately enable the convenience cell.
In Kaggle use the Output pane and Save Version. Clear notebook outputs before sharing
the notebook; return the small evidence files, never tokens, weights or images.
The first returned T4 run verified all official weights and FP32 checks, with ConvNeXt
BF16 failures. Complete acceptance, corrected-auto AMP and storage remain PENDING
until supported by a successful new report; fresh-session durability is a separate check.

In [ ]:
for name in ("preflight-report.json", "preflight-evidence.zip", "evidence-inventory.json"):
    path = OUTPUT_DIR / name
    print({"artifact": str(path), "exists": path.is_file()})
print(json.dumps({"status": report["status"], "coverage": report.get("coverage"),
                  "acceptance_complete": report.get("acceptance_complete"),
                  "acceptance_pending": report.get("acceptance_pending"),
                  "failures": report.get("failures", [])}, indent=2))
DOWNLOAD_COLAB_EVIDENCE = False
if CLOUD == "colab" and DOWNLOAD_COLAB_EVIDENCE:
    from google.colab import files
    for name in ("preflight-report.json", "preflight-evidence.zip"):
        files.download(str(OUTPUT_DIR / name))